# Pakistan Navy Eligibility Predictor
## Model Training Notebook

We use **only Pandas, NumPy and Scikit-Learn**.

Steps:
1. Load data
2. Preprocess
3. Train-test split
4. Encoding + Scaling (inside a Pipeline)
5. Train 5 models
6. Cross validation
7. Pick the best model
8. Save `best_model.pkl`

### 1. Import libraries

In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# The 5 models we must use
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

from sklearn.metrics import accuracy_score


### 2. Load the dataset
Make sure `navy_dataset.csv` is in the `data` folder.

In [2]:
df = pd.read_csv('../data/navy_dataset.csv')
print('Rows and columns:', df.shape)
df.head()


Rows and columns: (8500, 17)


,candidate_id,age,gender,height_cm,weight_kg,matric_percentage,inter_percentage,cgpa,physical_fitness_score,medical_status,marital_status,city,computer_skills,leadership_score,communication_score,branch_preference,eligibility_status
0,PN-16245,18,Female,166.2,60.8,44.00,53.82,0.00,84,Fit,Single,Karachi,Intermediate,77,78,Combat,Not Eligible
1,PN-16814,16,Male,172.3,73.7,55.61,61.02,2.43,67,Unfit,Married,Islamabad,Beginner,51,55,Medical Corps,Not Eligible
2,PN-17020,20,Male,172.6,66.9,72.39,71.43,2.45,56,Fit,Single,Lahore,Beginner,35,50,Naval Aviation,Eligible
3,PN-15538,25,Male,163.6,58.7,50.35,40.09,1.59,39,Fit,Single,Hyderabad,Advanced,57,53,Combat,Not Eligible
4,PN-16308,26,Male,171.4,96.5,71.14,77.99,2.87,55,Fit,Single,Karachi,Advanced,79,79,Logistics,Not Eligible


### 3. Quick look at the data

In [3]:
print(df.info())
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Target balance:')
print(df['eligibility_status'].value_counts())


<class 'pandas.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   candidate_id            8500 non-null   str    
 1   age                     8500 non-null   int64  
 2   gender                  8500 non-null   str    
 3   height_cm               8500 non-null   float64
 4   weight_kg               8500 non-null   float64
 5   matric_percentage       8500 non-null   float64
 6   inter_percentage        8500 non-null   float64
 7   cgpa                    8500 non-null   float64
 8   physical_fitness_score  8500 non-null   int64  
 9   medical_status          8500 non-null   str    
 10  marital_status          8500 non-null   str    
 11  city                    8500 non-null   str    
 12  computer_skills         8500 non-null   str    
 13  leadership_score        8500 non-null   int64  
 14  communication_score     8500 non-null   int64  
 15

### 4. Separate features (X) and target (y)
We drop `candidate_id` (just an id) and use `eligibility_status` as the target.

In [4]:
# Target column
y = df['eligibility_status']

# Input features = everything except the id and the target
X = df.drop(columns=['candidate_id', 'eligibility_status'])

print('Feature columns:')
print(list(X.columns))


Feature columns:
['age', 'gender', 'height_cm', 'weight_kg', 'matric_percentage', 'inter_percentage', 'cgpa', 'physical_fitness_score', 'medical_status', 'marital_status', 'city', 'computer_skills', 'leadership_score', 'communication_score', 'branch_preference']


### 5. Mark which columns are numeric and which are text
Numeric columns will be **scaled**. Text columns will be **encoded**.

In [5]:
numeric_cols = [
    'age', 'height_cm', 'weight_kg', 'matric_percentage',
    'inter_percentage', 'cgpa', 'physical_fitness_score',
    'leadership_score', 'communication_score'
]

categorical_cols = [
    'gender', 'medical_status', 'marital_status',
    'city', 'computer_skills', 'branch_preference'
]

print('Numeric:', numeric_cols)
print('Categorical:', categorical_cols)


Numeric: ['age', 'height_cm', 'weight_kg', 'matric_percentage', 'inter_percentage', 'cgpa', 'physical_fitness_score', 'leadership_score', 'communication_score']
Categorical: ['gender', 'medical_status', 'marital_status', 'city', 'computer_skills', 'branch_preference']


### 6. Build the preprocessing step
`ColumnTransformer` scales the numbers and one-hot encodes the text in one place.
Putting it inside a Pipeline means encoding + scaling are saved with the model.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)


### 7. Train-Test Split
80% for training, 20% for testing. `stratify=y` keeps the class balance.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', X_train.shape[0])
print('Test size :', X_test.shape[0])


Train size: 6800
Test size : 1700


### 8. Define the 5 models
Each model is wrapped in a Pipeline together with the preprocessor.

In [8]:
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

# Wrap every model in a pipeline: preprocess -> model
pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', model)
    ])
print('Pipelines ready:', list(pipelines.keys()))


Pipelines ready: ['KNN', 'Logistic Regression', 'Decision Tree', 'Bagging', 'AdaBoost']


### 9. Cross Validation + Training
We use 5-fold cross validation on the training data to compare models fairly,
then train each pipeline and check accuracy on the test set.

In [9]:
results = {}

for name, pipe in pipelines.items():
    # 5-fold cross validation (average accuracy)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()

    # Train on the full training set
    pipe.fit(X_train, y_train)

    # Check accuracy on the unseen test set
    test_acc = accuracy_score(y_test, pipe.predict(X_test))

    results[name] = {'cv_accuracy': cv_mean, 'test_accuracy': test_acc}
    print(f'{name:20s} | CV acc: {cv_mean:.3f} | Test acc: {test_acc:.3f}')


KNN                  | CV acc: 0.860 | Test acc: 0.863


Logistic Regression  | CV acc: 0.862 | Test acc: 0.868


Decision Tree        | CV acc: 0.972 | Test acc: 0.980


Bagging              | CV acc: 0.982 | Test acc: 0.982


AdaBoost             | CV acc: 0.949 | Test acc: 0.942


### 10. Compare results in a table

In [10]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('cv_accuracy', ascending=False)
results_df


,cv_accuracy,test_accuracy
Bagging,0.981618,0.982353
Decision Tree,0.971765,0.980000
AdaBoost,0.948971,0.942353
Logistic Regression,0.861912,0.867647
KNN,0.860147,0.862941


### 11. Pick the best model (highest cross-validation accuracy)

In [11]:
best_name = results_df.index[0]
best_model = pipelines[best_name]
print('Best model:', best_name)


Best model: Bagging


### 12. Save the best model
Saved as `../model/best_model.pkl` so the Flask app can load it.

In [12]:
joblib.dump(best_model, '../model/best_model.pkl')
print('Saved best_model.pkl ->', best_name)


Saved best_model.pkl -> Bagging


Training finished. Next, open **model_evaluation.ipynb** to see the
detailed scores (precision, recall, F1, confusion matrix).